# Hegselmann--Krause two-cluster latent-bridge experiment

This experiment is a structured stress test for the online learned controller.

The environment is deliberately constructed around one bounded-confidence
mechanism:

- 15 agents are split into two densely connected communities;
- each community has one very central hub;
- the only topological connection between the communities is the hub-to-hub edge;
- both communities are internally inside the HK confidence radius;
- the two hubs are initially farther apart than the confidence radius, so the
  cross-community edge is topologically present but dynamically inactive;
- campaign 0 is passive, exactly as in the main single-shot experiment;
- uniform control spreads the campaign budget across every node and should leave
  the hub bridge inactive for much of the horizon;
- the learned controller may concentrate control on the structurally important
  low-community hub, potentially activating the bridge much earlier.

The **primary comparison is learned nonlinear online control versus uniform
control**. No-control and true-graph centrality are retained only as diagnostic
references.

The notebook also contains a small robustness study around the handmade case:
five mild internal-weight variants crossed with five mild initial-opinion
variants, for 25 paired trials. Every generated trial must pass mechanism
assertions before it is admitted to the experiment.

## Running

The same notebook supports both worker and analysis modes.

- Run normally in Jupyter: it loads cached completed trials and produces plots.
- Set `HK_CLUSTER_SHARD_ID=0`, `1`, or `2`: it runs only that shard and writes
  each completed trial to the shared cache.
- The supplied CMD launcher starts all three shards in parallel.

No explicit random exploration is used: the epsilon schedule is zero after the
passive campaign, matching the conclusion of the main exploration study.

In [ ]:
from __future__ import annotations

import hashlib
import inspect
import json
import os
import random
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "opinion_dynamics").exists():
            return candidate
    raise RuntimeError(
        "Could not find repository root containing opinion_dynamics/."
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from rl_envs_forge.envs.network_graph.network_graph import NetworkGraph
from rl_envs_forge.envs.network_graph.graph_utils import (
    compute_eigenvector_centrality,
    compute_laplacian,
)
from opinion_dynamics.baseline import centrality_based_continuous_control
import opinion_dynamics.experiments.online_single_shot as online_single_shot_module

run_single_shot_online_identification = (
    online_single_shot_module.run_single_shot_online_identification
)

print("REPO_ROOT:", REPO_ROOT)
try:
    git_commit = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_ROOT,
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
except Exception:
    git_commit = "unknown"

print("Git commit:", git_commit)
print(
    "online_single_shot:",
    inspect.getsourcefile(online_single_shot_module),
)
print("NetworkGraph:", inspect.getsourcefile(NetworkGraph))

## Experiment configuration

In [ ]:
# ---------------------------------------------------------------------------
# Parallel worker mode
# ---------------------------------------------------------------------------
NUM_SHARDS = 3
_shard_env = os.environ.get("HK_CLUSTER_SHARD_ID")
WORKER_MODE = _shard_env is not None
SHARD_ID = int(_shard_env) if WORKER_MODE else None

if WORKER_MODE and SHARD_ID not in range(NUM_SHARDS):
    raise ValueError(
        f"HK_CLUSTER_SHARD_ID must be 0..{NUM_SHARDS - 1}; got {SHARD_ID}"
    )

TORCH_THREADS = int(os.environ.get("HK_CLUSTER_TORCH_THREADS", "3"))
torch.set_num_threads(max(1, TORCH_THREADS))
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

DEVICE = os.environ.get("HK_CLUSTER_DEVICE", "cpu").lower()
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA requested but unavailable.")

print("Mode:", "worker" if WORKER_MODE else "analysis")
if WORKER_MODE:
    print("Shard:", f"{SHARD_ID + 1}/{NUM_SHARDS}")
print("Device:", DEVICE)
print("Torch threads:", torch.get_num_threads())

# ---------------------------------------------------------------------------
# Handmade HK environment
# ---------------------------------------------------------------------------
STUDY_NAME = "hk_two_cluster_latent_bridge"
PIPELINE_VERSION = "2026-08-08-v1"

N = 15
LOW_CLUSTER = tuple(range(0, 7))
HIGH_CLUSTER = tuple(range(7, 15))
LOW_HUB = 0
HIGH_HUB = 7

TARGET = 1.0
HK_EPSILON = 0.25
HK_INCLUDE_SELF = True

NUM_CAMPAIGNS = 20
T_CAMPAIGN = 0.5
T_S = 0.1

MAX_U = 0.20
TOTAL_CONTROLLED_BUDGET = 6.0
B_CAMPAIGN = TOTAL_CONTROLLED_BUDGET / (NUM_CAMPAIGNS - 1)

LAMBDA_MIX = 0.70
EPSILON_SCHEDULE = [0.0] * NUM_CAMPAIGNS

# Same nonlinear identifier settings as the main single-shot study.
FIT_LR = 1e-3
FIT_MAX_STEPS = 1_000
FIT_MAE_STOP = 5e-4
FIT_BATCH_SIZE = 256
FIT_CHECK_EVERY = 200
IDENTIFIER_KWARGS = {"hidden_dim": 16}

# Graph construction.
HUB_LEAF_WEIGHT = 3.0
LEAF_RING_WEIGHT = 0.40
HUB_BRIDGE_WEIGHT = 3.0
WEIGHT_JITTER_FRAC = 0.05

# Handmade initial opinions.  The hubs are deliberately placed at the inner
# edges of their communities.  During campaign 0, each hub is pulled toward its
# own community, increasing the separation between the two community consensuses.
BASE_X0 = np.array(
    [
        0.35,              # low hub
        0.27, 0.28, 0.29, 0.30, 0.31, 0.32,
        0.64,              # high hub
        0.65, 0.66, 0.67, 0.68, 0.69, 0.70, 0.71,
    ],
    dtype=float,
)
OPINION_JITTER = 0.003

TOPOLOGY_VARIANT_SEEDS = [0, 1, 2, 3, 4]
OPINION_VARIANT_SEEDS = [0, 1, 2, 3, 4]
TRIAL_SPECS = [
    {
        "trial_id": i * len(OPINION_VARIANT_SEEDS) + j,
        "topology_variant_seed": topo_seed,
        "opinion_variant_seed": opinion_seed,
    }
    for i, topo_seed in enumerate(TOPOLOGY_VARIANT_SEEDS)
    for j, opinion_seed in enumerate(OPINION_VARIANT_SEEDS)
]

TRAIN_SEED_BASE = 880_000
RNG_SEED_BASE = 990_000

# Results/cache.
RESULTS_ROOT = (
    REPO_ROOT
    / "opinion_dynamics"
    / "experiments"
    / "results"
)
CACHE_ROOT = (
    RESULTS_ROOT
    / "_trial_cache"
    / STUDY_NAME
)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

ANALYSIS_DIR = RESULTS_ROOT / f"experiment_2026_08_08_{STUDY_NAME}"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print("Campaign budget:", B_CAMPAIGN)
print("Uniform allocation/node:", B_CAMPAIGN / N)
print("Number of paired trials:", len(TRIAL_SPECS))
print("Cache root:", CACHE_ROOT)

## Preserve HK parameters when the online runner clones the environment

The current online runner recreates an environment from a template.  This
notebook patches only those cloning hooks so that the handmade adjacency,
initial opinions, `hk_epsilon`, and `hk_include_self` survive the clone.

In [ ]:
def _copy_value(value: Any) -> Any:
    if isinstance(value, np.ndarray):
        return np.array(value, copy=True)
    if isinstance(value, list):
        return list(value)
    if isinstance(value, tuple):
        return tuple(value)
    return value


def online_env_kwargs_from_env(
    env: Any,
    *,
    t_campaign: float | None = None,
    t_s: float | None = None,
) -> dict[str, Any]:
    n = int(env.num_agents)

    kwargs = {
        "connectivity_matrix": np.array(
            env.connectivity_matrix,
            copy=True,
        ),
        "num_agents": n,
        "max_u": np.array(env.max_u, copy=True),
        "desired_opinion": float(env.desired_opinion),
        "t_campaign": float(
            env.t_campaign if t_campaign is None else t_campaign
        ),
        "t_s": float(env.t_s if t_s is None else t_s),
        "dynamics_model": str(env.dynamics_model),
        "initial_opinions": np.array(
            getattr(env, "initial_opinions", env.opinions),
            copy=True,
        ),
        "control_resistance": np.array(
            getattr(env, "control_resistance", np.zeros(n)),
            copy=True,
        ),
        "max_steps": int(getattr(env, "max_steps", 10_000)),
        "opinion_end_tolerance": float(
            getattr(env, "opinion_end_tolerance", 0.01)
        ),
        "control_beta": float(
            getattr(env, "control_beta", 0.4)
        ),
        "normalize_reward": bool(
            getattr(env, "normalize_reward", True)
        ),
        "terminal_reward": float(
            getattr(env, "terminal_reward", 0.0)
        ),
        "terminate_when_converged": bool(
            getattr(env, "terminate_when_converged", False)
        ),
        "budget": float(getattr(env, "budget", 1000.0)),
        "seed": (
            int(env.seed)
            if getattr(env, "seed", None) is not None
            else None
        ),
        "hk_epsilon": float(env.hk_epsilon),
        "hk_include_self": bool(env.hk_include_self),
    }

    signature = inspect.signature(env.__class__.__init__)
    accepts_kwargs = any(
        p.kind == inspect.Parameter.VAR_KEYWORD
        for p in signature.parameters.values()
    )

    if not accepts_kwargs:
        accepted = {
            name
            for name, p in signature.parameters.items()
            if name != "self"
            and p.kind in {
                inspect.Parameter.POSITIONAL_OR_KEYWORD,
                inspect.Parameter.KEYWORD_ONLY,
            }
        }
        required = {"hk_epsilon", "hk_include_self"}
        missing = required - accepted
        if missing:
            raise RuntimeError(
                "Installed NetworkGraph is too old for HK parameters: "
                f"{sorted(missing)}"
            )
        kwargs = {
            key: value
            for key, value in kwargs.items()
            if key in accepted
        }

    return kwargs


def online_make_env_from_template(
    env_template: Any,
    *,
    t_campaign: float | None = None,
    t_s: float | None = None,
) -> Any:
    return env_template.__class__(
        **online_env_kwargs_from_env(
            env_template,
            t_campaign=t_campaign,
            t_s=t_s,
        )
    )


online_single_shot_module.env_kwargs_from_env = (
    online_env_kwargs_from_env
)
online_single_shot_module.make_env_from_template = (
    online_make_env_from_template
)
run_single_shot_online_identification = (
    online_single_shot_module.run_single_shot_online_identification
)

print("Online environment-cloning hooks patched.")

## Handmade graph and initial-state family

In [ ]:
def _add_undirected_edge(
    raw: np.ndarray,
    i: int,
    j: int,
    weight: float,
) -> None:
    raw[i, j] = float(weight)
    raw[j, i] = float(weight)


def build_handmade_graph(
    variant_seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(10_000 + int(variant_seed))
    raw = np.zeros((N, N), dtype=float)

    def jitter(weight: float) -> float:
        multiplier = rng.uniform(
            1.0 - WEIGHT_JITTER_FRAC,
            1.0 + WEIGHT_JITTER_FRAC,
        )
        return float(weight * multiplier)

    # One strong hub per community.
    for node in LOW_CLUSTER:
        if node != LOW_HUB:
            _add_undirected_edge(
                raw,
                LOW_HUB,
                node,
                jitter(HUB_LEAF_WEIGHT),
            )

    for node in HIGH_CLUSTER:
        if node != HIGH_HUB:
            _add_undirected_edge(
                raw,
                HIGH_HUB,
                node,
                jitter(HUB_LEAF_WEIGHT),
            )

    # Weak leaf rings keep each community richly connected without challenging
    # the hub's structural dominance.
    for leaves in (
        [node for node in LOW_CLUSTER if node != LOW_HUB],
        [node for node in HIGH_CLUSTER if node != HIGH_HUB],
    ):
        for left, right in zip(leaves, leaves[1:] + leaves[:1]):
            _add_undirected_edge(
                raw,
                left,
                right,
                jitter(LEAF_RING_WEIGHT),
            )

    # This is the ONLY topological cross-community edge.
    _add_undirected_edge(
        raw,
        LOW_HUB,
        HIGH_HUB,
        jitter(HUB_BRIDGE_WEIGHT),
    )

    row_sums = raw.sum(axis=1, keepdims=True)
    if np.any(row_sums <= 0):
        raise RuntimeError("Every node must have a neighbor.")

    adjacency = raw / row_sums
    np.fill_diagonal(adjacency, 0.0)
    return raw, adjacency


def build_handmade_x0(opinion_seed: int) -> np.ndarray:
    rng = np.random.default_rng(20_000 + int(opinion_seed))
    x0 = BASE_X0 + rng.uniform(
        -OPINION_JITTER,
        OPINION_JITTER,
        size=N,
    )
    return np.clip(x0, 0.0, 1.0)


def make_env(
    adjacency: np.ndarray,
    x0: np.ndarray,
    *,
    seed: int,
) -> NetworkGraph:
    return NetworkGraph(
        connectivity_matrix=np.array(adjacency, copy=True),
        num_agents=N,
        max_u=MAX_U,
        desired_opinion=TARGET,
        t_campaign=T_CAMPAIGN,
        t_s=T_S,
        initial_opinions=np.array(x0, copy=True),
        control_resistance=np.zeros(N, dtype=float),
        dynamics_model="hegselmannkrause",
        hk_epsilon=HK_EPSILON,
        hk_include_self=HK_INCLUDE_SELF,
        budget=1000.0,
        max_steps=NUM_CAMPAIGNS + 20,
        opinion_end_tolerance=0.01,
        control_beta=0.4,
        normalize_reward=True,
        terminal_reward=0.0,
        terminate_when_converged=False,
        seed=int(seed),
    )


def centrality_from_A(adjacency: np.ndarray) -> np.ndarray:
    v = compute_eigenvector_centrality(
        compute_laplacian(np.asarray(adjacency, dtype=float))
    )
    v = np.asarray(v, dtype=float).reshape(-1)
    v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)

    if v.sum() < 0:
        v = -v

    v = np.maximum(v, 0.0)
    if v.sum() <= 1e-12:
        v = np.abs(v)

    if v.sum() <= 1e-12:
        return np.full(N, 1.0 / N)

    return v / v.sum()


def uniform_action(max_u: np.ndarray, budget: float) -> np.ndarray:
    return online_single_shot_module.uniform_budget_action(
        np.asarray(max_u, dtype=float).reshape(-1),
        float(budget),
    )


def true_graph_action(
    env: NetworkGraph,
    state: np.ndarray,
    v_true: np.ndarray,
) -> np.ndarray:
    # Use the same true-graph centrality baseline as the main experiments.
    old_state = np.array(env.opinions, copy=True)
    env.opinions = np.asarray(state, dtype=float).copy()
    action, _ = centrality_based_continuous_control(
        env,
        float(B_CAMPAIGN),
        v=np.asarray(v_true, dtype=float),
    )
    env.opinions = old_state
    return np.asarray(action, dtype=float)


def set_state(env: NetworkGraph, x0: np.ndarray) -> None:
    env.reset()
    env.opinions = np.asarray(x0, dtype=float).copy()
    if hasattr(env, "state"):
        try:
            env.state = np.asarray(x0, dtype=float).copy()
        except Exception:
            pass

In [ ]:
def cross_topological_edges(raw: np.ndarray) -> list[tuple[int, int]]:
    edges = []
    for i in LOW_CLUSTER:
        for j in HIGH_CLUSTER:
            if raw[i, j] > 0 or raw[j, i] > 0:
                edges.append((int(i), int(j)))
    return edges


def active_component_labels(
    adjacency: np.ndarray,
    state: np.ndarray,
) -> np.ndarray:
    state = np.asarray(state, dtype=float)
    undirected = (adjacency > 0) | (adjacency.T > 0)
    close = (
        np.abs(state[:, None] - state[None, :])
        <= HK_EPSILON
    )
    active = undirected & close
    np.fill_diagonal(active, False)

    labels = np.full(N, -1, dtype=int)
    component = 0

    for start in range(N):
        if labels[start] >= 0:
            continue

        stack = [start]
        labels[start] = component

        while stack:
            node = stack.pop()
            neighbors = np.flatnonzero(active[node])
            for neighbor in neighbors:
                if labels[neighbor] < 0:
                    labels[neighbor] = component
                    stack.append(int(neighbor))

        component += 1

    return labels


def number_active_components(
    adjacency: np.ndarray,
    state: np.ndarray,
) -> int:
    labels = active_component_labels(adjacency, state)
    return int(labels.max() + 1)


def passive_campaign_state(
    env: NetworkGraph,
    x0: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    set_state(env, x0)
    zero = np.zeros(N, dtype=float)
    x_next, _, _, _, info = env.step(zero)
    inter = np.asarray(info["intermediate_states"], dtype=float)
    return np.asarray(x_next, dtype=float), inter


def apply_impulse(
    state: np.ndarray,
    action: np.ndarray,
) -> np.ndarray:
    state = np.asarray(state, dtype=float)
    action = np.asarray(action, dtype=float)
    return action * TARGET + (1.0 - action) * state


def preflight_trial(spec: dict[str, int]) -> dict[str, Any]:
    raw, adjacency = build_handmade_graph(
        spec["topology_variant_seed"]
    )
    x0 = build_handmade_x0(spec["opinion_variant_seed"])
    env = make_env(
        adjacency,
        x0,
        seed=30_000 + spec["trial_id"],
    )

    assert cross_topological_edges(raw) == [(LOW_HUB, HIGH_HUB)]
    assert np.allclose(adjacency.sum(axis=1), 1.0)
    assert np.allclose(np.diag(adjacency), 0.0)

    v_true = centrality_from_A(adjacency)
    top_two = set(np.argsort(v_true)[-2:].tolist())
    assert top_two == {LOW_HUB, HIGH_HUB}, (
        "The two hubs must be the two most central nodes; "
        f"got {sorted(top_two)}"
    )

    initial_components = number_active_components(adjacency, x0)
    assert initial_components == 2, (
        f"Expected two initial confidence components, got {initial_components}"
    )

    x_passive, passive_inter = passive_campaign_state(env, x0)
    passive_components = number_active_components(
        adjacency,
        x_passive,
    )
    assert passive_components == 2, (
        "Passive campaign unexpectedly activated the bridge."
    )

    hub_gap_passive = float(
        abs(x_passive[LOW_HUB] - x_passive[HIGH_HUB])
    )
    assert hub_gap_passive > HK_EPSILON

    u_uniform = uniform_action(
        np.full(N, MAX_U),
        B_CAMPAIGN,
    )
    x_uniform_impulse = apply_impulse(x_passive, u_uniform)
    uniform_gap_after_impulse = float(
        abs(
            x_uniform_impulse[LOW_HUB]
            - x_uniform_impulse[HIGH_HUB]
        )
    )
    assert uniform_gap_after_impulse > HK_EPSILON, (
        "Uniform already activates the bridge in the first controlled "
        "campaign; the stress test is too easy."
    )

    u_oracle = true_graph_action(env, x_passive, v_true)
    x_oracle_impulse = apply_impulse(x_passive, u_oracle)
    oracle_gap_after_impulse = float(
        abs(
            x_oracle_impulse[LOW_HUB]
            - x_oracle_impulse[HIGH_HUB]
        )
    )
    assert oracle_gap_after_impulse <= HK_EPSILON, (
        "True-graph concentrated control failed to activate the bridge. "
        "The handmade trial does not encode the intended mechanism."
    )

    assert u_oracle[LOW_HUB] > 0.0, (
        "True-graph controller did not target the low-community hub."
    )

    return {
        **spec,
        "initial_components": initial_components,
        "passive_components": passive_components,
        "initial_hub_gap": float(
            abs(x0[LOW_HUB] - x0[HIGH_HUB])
        ),
        "passive_hub_gap": hub_gap_passive,
        "uniform_gap_after_first_impulse": (
            uniform_gap_after_impulse
        ),
        "oracle_gap_after_first_impulse": (
            oracle_gap_after_impulse
        ),
        "oracle_low_hub_allocation": float(
            u_oracle[LOW_HUB]
        ),
        "oracle_high_hub_allocation": float(
            u_oracle[HIGH_HUB]
        ),
        "low_hub_centrality": float(v_true[LOW_HUB]),
        "high_hub_centrality": float(v_true[HIGH_HUB]),
    }


preflight_rows = [
    preflight_trial(spec)
    for spec in TRIAL_SPECS
]
preflight_df = pd.DataFrame(preflight_rows)

print("All 25 handmade variants passed the mechanism preflight.")
display(preflight_df.round(5))

## Visualize the canonical handmade trial

In [ ]:
canonical_spec = TRIAL_SPECS[0]
canonical_raw, canonical_A = build_handmade_graph(
    canonical_spec["topology_variant_seed"]
)
canonical_x0 = build_handmade_x0(
    canonical_spec["opinion_variant_seed"]
)
canonical_v = centrality_from_A(canonical_A)

# Manual positions: hubs near the center, leaves spread outward.
positions: dict[int, tuple[float, float]] = {
    LOW_HUB: (-0.8, 0.0),
    HIGH_HUB: (0.8, 0.0),
}

low_leaves = [i for i in LOW_CLUSTER if i != LOW_HUB]
high_leaves = [i for i in HIGH_CLUSTER if i != HIGH_HUB]

for idx, node in enumerate(low_leaves):
    angle = 2.0 * np.pi * idx / len(low_leaves)
    positions[node] = (
        -2.1 + 0.65 * np.cos(angle),
        0.85 * np.sin(angle),
    )

for idx, node in enumerate(high_leaves):
    angle = 2.0 * np.pi * idx / len(high_leaves)
    positions[node] = (
        2.1 + 0.65 * np.cos(angle),
        0.85 * np.sin(angle),
    )

fig, ax = plt.subplots(figsize=(12, 6))

for i in range(N):
    for j in range(i + 1, N):
        if canonical_raw[i, j] <= 0:
            continue
        x_i, y_i = positions[i]
        x_j, y_j = positions[j]
        linewidth = (
            3.0
            if {i, j} == {LOW_HUB, HIGH_HUB}
            else 1.0
        )
        linestyle = (
            "--"
            if {i, j} == {LOW_HUB, HIGH_HUB}
            else "-"
        )
        ax.plot(
            [x_i, x_j],
            [y_i, y_j],
            linewidth=linewidth,
            linestyle=linestyle,
            alpha=0.65,
        )

for node in range(N):
    x_pos, y_pos = positions[node]
    is_hub = node in {LOW_HUB, HIGH_HUB}
    ax.scatter(
        [x_pos],
        [y_pos],
        s=850 if is_hub else 420,
        zorder=3,
    )
    ax.text(
        x_pos,
        y_pos,
        (
            f"{node}\n"
            f"x0={canonical_x0[node]:.3f}\n"
            f"v={canonical_v[node]:.3f}"
        ),
        ha="center",
        va="center",
        fontsize=8 if is_hub else 7,
        zorder=4,
    )

ax.set_title(
    "Handmade HK topology: only the two hubs connect the communities"
)
ax.axis("off")
fig.tight_layout()
fig.savefig(
    ANALYSIS_DIR / "handmade_topology.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

## Cache identity

The cache is treated as persistent scientific result storage.  Trial identity
contains both the scientific configuration and hashes of the critical source
files.  A relevant implementation change therefore creates a different cache
identity instead of silently reusing an older trajectory.

In [ ]:
def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def source_hash(obj: Any) -> str:
    path_string = inspect.getsourcefile(obj)
    if not path_string:
        return "unavailable"
    path = Path(path_string)
    if not path.exists():
        return "missing"
    return sha256_bytes(path.read_bytes())


SOURCE_HASHES = {
    "online_single_shot": source_hash(
        online_single_shot_module
    ),
    "NetworkGraph": source_hash(NetworkGraph),
}

scientific_config = {
    "pipeline_version": PIPELINE_VERSION,
    "N": N,
    "low_cluster": LOW_CLUSTER,
    "high_cluster": HIGH_CLUSTER,
    "low_hub": LOW_HUB,
    "high_hub": HIGH_HUB,
    "target": TARGET,
    "hk_epsilon": HK_EPSILON,
    "hk_include_self": HK_INCLUDE_SELF,
    "num_campaigns": NUM_CAMPAIGNS,
    "t_campaign": T_CAMPAIGN,
    "t_s": T_S,
    "max_u": MAX_U,
    "total_controlled_budget": TOTAL_CONTROLLED_BUDGET,
    "B_campaign": B_CAMPAIGN,
    "lambda_mix": LAMBDA_MIX,
    "epsilon_schedule": EPSILON_SCHEDULE,
    "fit_lr": FIT_LR,
    "fit_max_steps": FIT_MAX_STEPS,
    "fit_mae_stop": FIT_MAE_STOP,
    "fit_batch_size": FIT_BATCH_SIZE,
    "fit_check_every": FIT_CHECK_EVERY,
    "identifier_kwargs": IDENTIFIER_KWARGS,
    "hub_leaf_weight": HUB_LEAF_WEIGHT,
    "leaf_ring_weight": LEAF_RING_WEIGHT,
    "hub_bridge_weight": HUB_BRIDGE_WEIGHT,
    "weight_jitter_frac": WEIGHT_JITTER_FRAC,
    "base_x0": BASE_X0.tolist(),
    "opinion_jitter": OPINION_JITTER,
}

CONFIG_HASH = sha256_bytes(
    json.dumps(
        scientific_config,
        sort_keys=True,
        default=str,
    ).encode("utf-8")
)
IMPLEMENTATION_HASH = sha256_bytes(
    json.dumps(
        SOURCE_HASHES,
        sort_keys=True,
    ).encode("utf-8")
)

print("Config hash:", CONFIG_HASH[:16])
print("Implementation hash:", IMPLEMENTATION_HASH[:16])
display(pd.DataFrame([SOURCE_HASHES]))

## Rollout and metric helpers

In [ ]:
def set_global_seed(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def rollout_fixed_policy(
    env_template: NetworkGraph,
    x0: np.ndarray,
    *,
    policy: str,
    v_true: np.ndarray | None = None,
) -> dict[str, Any]:
    env = online_make_env_from_template(env_template)
    set_state(env, x0)

    state = np.asarray(x0, dtype=float).copy()
    states = [state.copy()]
    actions = []
    rewards = []
    intermediate_states_list = []

    max_u = np.asarray(env.max_u, dtype=float).reshape(-1)

    for campaign in range(NUM_CAMPAIGNS):
        if campaign == 0 or policy == "no_control":
            action = np.zeros(N, dtype=float)
        elif policy == "uniform":
            action = uniform_action(max_u, B_CAMPAIGN)
        elif policy == "true_graph":
            if v_true is None:
                raise ValueError("v_true required")
            action = true_graph_action(env, state, v_true)
        else:
            raise ValueError(policy)

        next_state, reward, done, trunc, info = env.step(action)

        states.append(np.asarray(next_state, dtype=float).copy())
        actions.append(np.asarray(action, dtype=float).copy())
        rewards.append(float(reward))
        intermediate_states_list.append(
            np.asarray(
                info["intermediate_states"],
                dtype=float,
            ).copy()
        )

        state = np.asarray(next_state, dtype=float).copy()

        if done or trunc:
            break

    return {
        "policy": policy,
        "states": np.asarray(states, dtype=float),
        "actions": np.asarray(actions, dtype=float),
        "rewards": np.asarray(rewards, dtype=float),
        "intermediate_states_list": intermediate_states_list,
    }


def rollout_learned(
    env_template: NetworkGraph,
    x0: np.ndarray,
    *,
    trial_id: int,
) -> dict[str, Any]:
    train_seed = TRAIN_SEED_BASE + int(trial_id)
    rng_seed = RNG_SEED_BASE + int(trial_id)

    set_global_seed(train_seed)

    out = run_single_shot_online_identification(
        env_template,
        x0=np.asarray(x0, dtype=float),
        random_initial_opinions=False,
        num_campaigns_total=NUM_CAMPAIGNS,
        t_campaign=T_CAMPAIGN,
        t_s=T_S,
        B_campaign=B_CAMPAIGN,
        lambda_mix=LAMBDA_MIX,
        exploration_campaigns=0,
        epsilon_schedule=EPSILON_SCHEDULE,
        lr=FIT_LR,
        l2_lambda=0.0,
        fit_max_steps=FIT_MAX_STEPS,
        fit_mae_stop=FIT_MAE_STOP,
        fit_batch_size=FIT_BATCH_SIZE,
        fit_check_every=FIT_CHECK_EVERY,
        identifier_kwargs=IDENTIFIER_KWARGS,
        device=DEVICE,
        rng_seed=rng_seed,
        suppress_fit_logs=True,
    )
    out = dict(out)
    out["policy"] = "learned_nonlinear"
    return out


def fine_trajectory(
    rollout: dict[str, Any],
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    times = [0.0]
    states = [np.asarray(rollout["states"][0], dtype=float)]
    campaigns = [-1]

    for campaign, intermediate in enumerate(
        rollout["intermediate_states_list"]
    ):
        inter = np.asarray(intermediate, dtype=float)
        campaign_start = campaign * T_CAMPAIGN

        # post-impulse state at the same campaign boundary
        times.append(campaign_start)
        states.append(inter[0])
        campaigns.append(campaign)

        for step in range(1, inter.shape[0]):
            times.append(campaign_start + step * T_S)
            states.append(inter[step])
            campaigns.append(campaign)

    return (
        np.asarray(times, dtype=float),
        np.asarray(states, dtype=float),
        np.asarray(campaigns, dtype=int),
    )


def bridge_metrics(
    rollout: dict[str, Any],
    adjacency: np.ndarray,
) -> dict[str, Any]:
    times, fine_states, fine_campaigns = fine_trajectory(rollout)

    hub_gap = np.abs(
        fine_states[:, LOW_HUB] - fine_states[:, HIGH_HUB]
    )
    active = hub_gap <= HK_EPSILON

    first_index = (
        int(np.flatnonzero(active)[0])
        if active.any()
        else None
    )

    component_counts = np.array(
        [
            number_active_components(adjacency, state)
            for state in fine_states
        ],
        dtype=int,
    )
    connected = component_counts == 1
    first_connected_index = (
        int(np.flatnonzero(connected)[0])
        if connected.any()
        else None
    )

    return {
        "first_bridge_time": (
            float(times[first_index])
            if first_index is not None
            else np.nan
        ),
        "first_bridge_campaign": (
            int(fine_campaigns[first_index])
            if first_index is not None
            else np.nan
        ),
        "first_connected_time": (
            float(times[first_connected_index])
            if first_connected_index is not None
            else np.nan
        ),
        "first_connected_campaign": (
            int(fine_campaigns[first_connected_index])
            if first_connected_index is not None
            else np.nan
        ),
        "final_hub_gap": float(hub_gap[-1]),
        "final_components": int(component_counts[-1]),
    }


def summarize_rollout(
    rollout: dict[str, Any],
    adjacency: np.ndarray,
) -> dict[str, Any]:
    states = np.asarray(rollout["states"], dtype=float)
    final = states[-1]
    means = states.mean(axis=1)

    bridge = bridge_metrics(rollout, adjacency)

    return {
        "policy": rollout["policy"],
        "final_mean": float(final.mean()),
        "mean_over_campaigns": float(means.mean()),
        "final_min": float(final.min()),
        "final_max": float(final.max()),
        "final_low_cluster_mean": float(
            final[list(LOW_CLUSTER)].mean()
        ),
        "final_high_cluster_mean": float(
            final[list(HIGH_CLUSTER)].mean()
        ),
        "final_cluster_mean_gap": float(
            abs(
                final[list(LOW_CLUSTER)].mean()
                - final[list(HIGH_CLUSTER)].mean()
            )
        ),
        **bridge,
    }

## Per-trial persistent cache

In [ ]:
POLICY_ORDER = [
    "no_control",
    "uniform",
    "true_graph",
    "learned_nonlinear",
]


def trial_cache_key(spec: dict[str, int]) -> str:
    payload = {
        "spec": spec,
        "config_hash": CONFIG_HASH,
        "implementation_hash": IMPLEMENTATION_HASH,
    }
    return sha256_bytes(
        json.dumps(
            payload,
            sort_keys=True,
        ).encode("utf-8")
    )


def trial_cache_dir(spec: dict[str, int]) -> Path:
    key = trial_cache_key(spec)
    return CACHE_ROOT / f"trial_{spec['trial_id']:02d}_{key[:16]}"


def cache_complete(spec: dict[str, int]) -> bool:
    directory = trial_cache_dir(spec)
    manifest = directory / "manifest.json"
    arrays = directory / "rollouts.npz"
    summary = directory / "summary.json"

    if not (manifest.exists() and arrays.exists() and summary.exists()):
        return False

    try:
        payload = json.loads(manifest.read_text(encoding="utf-8"))
    except Exception:
        return False

    return (
        payload.get("status") == "success"
        and payload.get("config_hash") == CONFIG_HASH
        and payload.get("implementation_hash") == IMPLEMENTATION_HASH
        and payload.get("trial_key") == trial_cache_key(spec)
    )


def save_trial_cache(
    spec: dict[str, int],
    rollouts: dict[str, dict[str, Any]],
    summary_rows: list[dict[str, Any]],
    preflight_row: dict[str, Any],
) -> Path:
    directory = trial_cache_dir(spec)
    directory.mkdir(parents=True, exist_ok=True)

    arrays: dict[str, np.ndarray] = {}
    for policy, rollout in rollouts.items():
        arrays[f"{policy}__states"] = np.asarray(
            rollout["states"],
            dtype=float,
        )
        arrays[f"{policy}__actions"] = np.asarray(
            rollout["actions"],
            dtype=float,
        )

        for campaign, inter in enumerate(
            rollout["intermediate_states_list"]
        ):
            arrays[
                f"{policy}__intermediate__{campaign:02d}"
            ] = np.asarray(inter, dtype=float)

    npz_tmp = directory / "rollouts.tmp"
    npz_final = directory / "rollouts.npz"
    with npz_tmp.open("wb") as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(npz_tmp, npz_final)

    summary_tmp = directory / "summary.json.tmp"
    summary_final = directory / "summary.json"
    summary_tmp.write_text(
        json.dumps(summary_rows, indent=2),
        encoding="utf-8",
    )
    os.replace(summary_tmp, summary_final)

    preflight_tmp = directory / "preflight.json.tmp"
    preflight_final = directory / "preflight.json"
    preflight_tmp.write_text(
        json.dumps(preflight_row, indent=2),
        encoding="utf-8",
    )
    os.replace(preflight_tmp, preflight_final)

    manifest_payload = {
        "status": "success",
        "study_name": STUDY_NAME,
        "pipeline_version": PIPELINE_VERSION,
        "trial_key": trial_cache_key(spec),
        "trial_spec": spec,
        "config_hash": CONFIG_HASH,
        "implementation_hash": IMPLEMENTATION_HASH,
        "source_hashes": SOURCE_HASHES,
        "git_commit": git_commit,
        "completed_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
    }

    manifest_tmp = directory / "manifest.json.tmp"
    manifest_final = directory / "manifest.json"
    manifest_tmp.write_text(
        json.dumps(
            manifest_payload,
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )
    os.replace(manifest_tmp, manifest_final)

    return directory


def load_trial_cache(
    spec: dict[str, int],
) -> tuple[
    list[dict[str, Any]],
    dict[str, dict[str, Any]],
]:
    if not cache_complete(spec):
        raise FileNotFoundError(
            f"Missing complete cache for {spec}"
        )

    directory = trial_cache_dir(spec)
    summary_rows = json.loads(
        (directory / "summary.json").read_text(
            encoding="utf-8"
        )
    )

    loaded = np.load(
        directory / "rollouts.npz",
        allow_pickle=False,
    )

    rollouts: dict[str, dict[str, Any]] = {}
    for policy in POLICY_ORDER:
        prefix = f"{policy}__"
        states = loaded[f"{prefix}states"]
        actions = loaded[f"{prefix}actions"]

        intermediate_keys = sorted(
            key
            for key in loaded.files
            if key.startswith(
                f"{policy}__intermediate__"
            )
        )

        rollouts[policy] = {
            "policy": policy,
            "states": states,
            "actions": actions,
            "intermediate_states_list": [
                loaded[key]
                for key in intermediate_keys
            ],
        }

    return summary_rows, rollouts

## Run assigned trials

In [ ]:
def run_one_trial(
    spec: dict[str, int],
) -> tuple[
    list[dict[str, Any]],
    dict[str, dict[str, Any]],
]:
    if cache_complete(spec):
        print(
            f"[trial {spec['trial_id']:02d}] cache hit"
        )
        return load_trial_cache(spec)

    print(
        f"[trial {spec['trial_id']:02d}] running "
        f"topology={spec['topology_variant_seed']} "
        f"opinions={spec['opinion_variant_seed']}"
    )

    preflight_row = preflight_trial(spec)

    raw, adjacency = build_handmade_graph(
        spec["topology_variant_seed"]
    )
    x0 = build_handmade_x0(
        spec["opinion_variant_seed"]
    )
    env_template = make_env(
        adjacency,
        x0,
        seed=40_000 + spec["trial_id"],
    )
    v_true = centrality_from_A(adjacency)

    rollouts = {
        "no_control": rollout_fixed_policy(
            env_template,
            x0,
            policy="no_control",
        ),
        "uniform": rollout_fixed_policy(
            env_template,
            x0,
            policy="uniform",
        ),
        "true_graph": rollout_fixed_policy(
            env_template,
            x0,
            policy="true_graph",
            v_true=v_true,
        ),
        "learned_nonlinear": rollout_learned(
            env_template,
            x0,
            trial_id=spec["trial_id"],
        ),
    }

    summary_rows = []
    for policy, rollout in rollouts.items():
        row = summarize_rollout(
            rollout,
            adjacency,
        )
        row.update(spec)
        row["train_seed"] = (
            TRAIN_SEED_BASE + spec["trial_id"]
            if policy == "learned_nonlinear"
            else None
        )
        summary_rows.append(row)

    save_trial_cache(
        spec,
        rollouts,
        summary_rows,
        preflight_row,
    )

    return summary_rows, rollouts


if WORKER_MODE:
    assigned_specs = [
        spec
        for spec in TRIAL_SPECS
        if spec["trial_id"] % NUM_SHARDS == SHARD_ID
    ]

    print(
        "Assigned trial IDs:",
        [spec["trial_id"] for spec in assigned_specs],
    )

    worker_rows = []
    t0 = time.perf_counter()

    for spec in assigned_specs:
        rows, _ = run_one_trial(spec)
        worker_rows.extend(rows)

    elapsed = time.perf_counter() - t0

    worker_df = pd.DataFrame(worker_rows)
    worker_output = (
        ANALYSIS_DIR
        / f"worker_shard_{SHARD_ID}_summary.csv"
    )
    worker_df.to_csv(worker_output, index=False)

    marker = (
        ANALYSIS_DIR
        / f"SHARD_{SHARD_ID}_COMPLETE.json"
    )
    marker.write_text(
        json.dumps(
            {
                "status": "success",
                "shard_id": SHARD_ID,
                "trial_ids": [
                    spec["trial_id"]
                    for spec in assigned_specs
                ],
                "elapsed_s": elapsed,
                "config_hash": CONFIG_HASH,
                "implementation_hash": IMPLEMENTATION_HASH,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    print(
        f"Shard {SHARD_ID} complete in {elapsed / 60:.1f} min."
    )
else:
    print(
        "Analysis mode: no trials are executed in this cell. "
        "Use the CMD launcher for the parallel run."
    )

## Load the completed paired experiment

This section runs only in analysis mode.  It refuses to analyze a partial
experiment: all 25 paired trial caches must be present.

In [ ]:
if not WORKER_MODE:
    missing_trials = [
        spec["trial_id"]
        for spec in TRIAL_SPECS
        if not cache_complete(spec)
    ]

    if missing_trials:
        raise RuntimeError(
            "Experiment is incomplete. Missing trial caches: "
            f"{missing_trials}. Run the three-shard launcher first."
        )

    all_summary_rows = []
    all_rollouts: dict[
        int,
        dict[str, dict[str, Any]],
    ] = {}

    for spec in TRIAL_SPECS:
        rows, rollouts = load_trial_cache(spec)
        all_summary_rows.extend(rows)
        all_rollouts[spec["trial_id"]] = rollouts

    summary_df = pd.DataFrame(all_summary_rows)

    expected_rows = len(TRIAL_SPECS) * len(POLICY_ORDER)
    assert len(summary_df) == expected_rows
    assert not summary_df.duplicated(
        subset=["trial_id", "policy"]
    ).any()

    print("Loaded complete experiment.")
    print("Summary rows:", len(summary_df))
    display(
        summary_df.groupby("policy")[
            [
                "final_mean",
                "mean_over_campaigns",
                "first_bridge_campaign",
                "final_cluster_mean_gap",
            ]
        ].mean().round(4)
    )

## Primary paired comparison: learned nonlinear vs uniform

In [ ]:
if not WORKER_MODE:
    learned = (
        summary_df[
            summary_df["policy"] == "learned_nonlinear"
        ]
        .set_index("trial_id")
        .sort_index()
    )
    uniform = (
        summary_df[
            summary_df["policy"] == "uniform"
        ]
        .set_index("trial_id")
        .sort_index()
    )
    oracle = (
        summary_df[
            summary_df["policy"] == "true_graph"
        ]
        .set_index("trial_id")
        .sort_index()
    )

    assert learned.index.equals(uniform.index)
    assert learned.index.equals(oracle.index)

    paired = pd.DataFrame(
        {
            "trial_id": learned.index,
            "final_mean_learned": learned["final_mean"].to_numpy(),
            "final_mean_uniform": uniform["final_mean"].to_numpy(),
            "final_mean_oracle": oracle["final_mean"].to_numpy(),
            "gap_learned_minus_uniform": (
                learned["final_mean"].to_numpy()
                - uniform["final_mean"].to_numpy()
            ),
            "gap_oracle_minus_uniform": (
                oracle["final_mean"].to_numpy()
                - uniform["final_mean"].to_numpy()
            ),
            "mean_over_campaigns_gap": (
                learned["mean_over_campaigns"].to_numpy()
                - uniform["mean_over_campaigns"].to_numpy()
            ),
            "bridge_campaign_learned": (
                learned["first_bridge_campaign"].to_numpy()
            ),
            "bridge_campaign_uniform": (
                uniform["first_bridge_campaign"].to_numpy()
            ),
            "bridge_campaign_oracle": (
                oracle["first_bridge_campaign"].to_numpy()
            ),
        }
    )


    def mean_ci95(values: np.ndarray) -> tuple[float, float]:
        values = np.asarray(values, dtype=float)
        values = values[np.isfinite(values)]
        if len(values) == 0:
            return np.nan, np.nan
        mean = float(values.mean())
        if len(values) == 1:
            return mean, 0.0
        half = float(
            1.96 * values.std(ddof=1) / np.sqrt(len(values))
        )
        return mean, half


    gap_mean, gap_ci = mean_ci95(
        paired["gap_learned_minus_uniform"].to_numpy()
    )
    auc_mean, auc_ci = mean_ci95(
        paired["mean_over_campaigns_gap"].to_numpy()
    )

    result_overview = pd.DataFrame(
        [
            {
                "comparison": "learned - uniform final mean",
                "mean": gap_mean,
                "ci95_half_width": gap_ci,
                "win_rate": float(
                    (
                        paired["gap_learned_minus_uniform"]
                        > 0
                    ).mean()
                ),
            },
            {
                "comparison": "learned - uniform mean over campaigns",
                "mean": auc_mean,
                "ci95_half_width": auc_ci,
                "win_rate": float(
                    (
                        paired["mean_over_campaigns_gap"]
                        > 0
                    ).mean()
                ),
            },
        ]
    )

    display(result_overview.round(5))
    display(paired.round(5))

    summary_df.to_csv(
        ANALYSIS_DIR / "summary_all_policies.csv",
        index=False,
    )
    paired.to_csv(
        ANALYSIS_DIR / "paired_learned_vs_uniform.csv",
        index=False,
    )
    preflight_df.to_csv(
        ANALYSIS_DIR / "mechanism_preflight.csv",
        index=False,
    )

## Aggregate result figures

In [ ]:
if not WORKER_MODE:
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.axhline(0.0, linewidth=1.0, linestyle="--")
    ax.scatter(
        paired["trial_id"],
        paired["gap_learned_minus_uniform"],
        label="learned - uniform",
    )
    ax.set_xlabel("paired handmade trial")
    ax.set_ylabel("final mean-opinion gain")
    ax.set_title(
        "HK latent-bridge stress test: learned control vs uniform"
    )
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(
        ANALYSIS_DIR / "paired_final_mean_gain.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

    bridge_plot = paired[
        [
            "trial_id",
            "bridge_campaign_learned",
            "bridge_campaign_uniform",
            "bridge_campaign_oracle",
        ]
    ].copy()

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.scatter(
        bridge_plot["trial_id"],
        bridge_plot["bridge_campaign_learned"],
        label="learned nonlinear",
    )
    ax.scatter(
        bridge_plot["trial_id"],
        bridge_plot["bridge_campaign_uniform"],
        label="uniform",
    )
    ax.scatter(
        bridge_plot["trial_id"],
        bridge_plot["bridge_campaign_oracle"],
        label="true graph",
    )
    ax.set_xlabel("paired handmade trial")
    ax.set_ylabel("first bridge-active campaign")
    ax.set_title("When does the latent hub-to-hub bridge activate?")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(
        ANALYSIS_DIR / "bridge_activation_campaign.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

## All individual trajectories for the canonical trial

Every node is drawn separately.  Solid lines are the low community, dashed
lines are the high community, and the two hub trajectories are thicker.

The first figure is the primary learned-vs-uniform comparison.  Separate
diagnostic figures for no control and the true-graph controller follow.

In [ ]:
def plot_individual_trajectories(
    rollout: dict[str, Any],
    *,
    title: str,
    filename: str,
) -> None:
    times, fine_states, _ = fine_trajectory(rollout)

    fig, ax = plt.subplots(figsize=(13, 7))

    for node in range(N):
        in_low = node in LOW_CLUSTER
        is_hub = node in {LOW_HUB, HIGH_HUB}

        ax.plot(
            times,
            fine_states[:, node],
            linestyle="-" if in_low else "--",
            linewidth=3.0 if is_hub else 1.15,
            label=(
                f"node {node}"
                + (" | low" if in_low else " | high")
                + (" hub" if is_hub else "")
            ),
        )

    ax.axhline(
        TARGET,
        linestyle=":",
        linewidth=1.2,
        label="target",
    )
    ax.set_ylim(0.2, 1.02)
    ax.set_xlabel("time")
    ax.set_ylabel("opinion")
    ax.set_title(title)
    ax.grid(alpha=0.25)
    ax.legend(
        bbox_to_anchor=(1.02, 1.0),
        loc="upper left",
        fontsize=8,
    )
    fig.tight_layout()
    fig.savefig(
        ANALYSIS_DIR / filename,
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)


if not WORKER_MODE:
    canonical_rollouts = all_rollouts[0]

    plot_individual_trajectories(
        canonical_rollouts["uniform"],
        title="All node trajectories — uniform control",
        filename="canonical_individual_trajectories_uniform.png",
    )
    plot_individual_trajectories(
        canonical_rollouts["learned_nonlinear"],
        title="All node trajectories — learned nonlinear control",
        filename="canonical_individual_trajectories_learned.png",
    )
    plot_individual_trajectories(
        canonical_rollouts["true_graph"],
        title="All node trajectories — true-graph diagnostic",
        filename="canonical_individual_trajectories_true_graph.png",
    )
    plot_individual_trajectories(
        canonical_rollouts["no_control"],
        title="All node trajectories — no control",
        filename="canonical_individual_trajectories_no_control.png",
    )

## Direct learned-versus-uniform trajectory comparison

In [ ]:
if not WORKER_MODE:
    canonical_rollouts = all_rollouts[0]

    fig, ax = plt.subplots(figsize=(11, 5.5))

    for policy, label in [
        ("uniform", "uniform"),
        ("learned_nonlinear", "learned nonlinear"),
        ("true_graph", "true graph"),
        ("no_control", "no control"),
    ]:
        rollout = canonical_rollouts[policy]
        times, fine_states, _ = fine_trajectory(rollout)
        ax.plot(
            times,
            fine_states.mean(axis=1),
            linewidth=2.0,
            label=label,
        )

    ax.set_xlabel("time")
    ax.set_ylabel("population mean opinion")
    ax.set_title(
        "Canonical two-cluster HK trial — population mean trajectories"
    )
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(
        ANALYSIS_DIR / "canonical_mean_trajectory_comparison.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(11, 5.5))

    for policy, label in [
        ("uniform", "uniform"),
        ("learned_nonlinear", "learned nonlinear"),
        ("true_graph", "true graph"),
    ]:
        rollout = canonical_rollouts[policy]
        times, fine_states, _ = fine_trajectory(rollout)
        gap = np.abs(
            fine_states[:, LOW_HUB]
            - fine_states[:, HIGH_HUB]
        )
        ax.plot(times, gap, linewidth=2.0, label=label)

    ax.axhline(
        HK_EPSILON,
        linestyle="--",
        linewidth=1.3,
        label="HK epsilon",
    )
    ax.set_xlabel("time")
    ax.set_ylabel("hub opinion gap")
    ax.set_title(
        "Canonical two-cluster HK trial — latent bridge activation"
    )
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(
        ANALYSIS_DIR / "canonical_hub_gap_comparison.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

## Where does the learned controller spend its budget?

In [ ]:
if not WORKER_MODE:
    learned_rollout = all_rollouts[0]["learned_nonlinear"]
    learned_actions = np.asarray(
        learned_rollout["actions"],
        dtype=float,
    )

    fig, ax = plt.subplots(figsize=(12, 5))
    image = ax.imshow(
        learned_actions.T,
        aspect="auto",
        interpolation="nearest",
    )
    ax.set_xlabel("campaign")
    ax.set_ylabel("node")
    ax.set_xticks(np.arange(learned_actions.shape[0]))
    ax.set_yticks(np.arange(N))
    ax.set_title(
        "Canonical learned-controller allocations "
        "(hubs are nodes 0 and 7)"
    )
    fig.colorbar(image, ax=ax, label="control allocation")
    fig.tight_layout()
    fig.savefig(
        ANALYSIS_DIR / "canonical_learned_action_heatmap.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

    hub_allocations = pd.DataFrame(
        {
            "campaign": np.arange(
                learned_actions.shape[0]
            ),
            "low_hub": learned_actions[:, LOW_HUB],
            "high_hub": learned_actions[:, HIGH_HUB],
            "all_other_nodes": (
                learned_actions.sum(axis=1)
                - learned_actions[:, LOW_HUB]
                - learned_actions[:, HIGH_HUB]
            ),
        }
    )
    display(hub_allocations.round(5))
    hub_allocations.to_csv(
        ANALYSIS_DIR / "canonical_hub_allocations.csv",
        index=False,
    )

## Interpretation

The experiment has three possible scientifically useful outcomes.

1. **Learned control activates the bridge substantially earlier than uniform
   and improves the final/average opinion.**  
   This is the intended demonstration: learned structural targeting exploits a
   bounded-confidence bottleneck that uniform allocation reaches only slowly.

2. **The true-graph controller succeeds but the learned controller does not.**  
   The environment contains the desired control opportunity, but one passive
   campaign does not provide enough information for the current identifier /
   centrality conversion to exploit it.

3. **Uniform also activates the bridge very early.**  
   The handmade geometry is too easy.  Tighten the test by increasing the
   post-passive hub gap or decreasing `HK_EPSILON`, while retaining the
   preflight requirement that true-graph concentrated control can activate the
   bridge in one controlled campaign.

Do not change parameters solely to maximize the learned-minus-uniform result
after seeing the 25-trial outcome.  The preflight mechanism conditions are the
appropriate basis for defining the stress test.